# Richards-Wolf Vector Diffraction Demonstration

This notebook demonstrates the Richards-Wolf vectorial diffraction theory for high-NA focusing.

The Richards-Wolf theory computes the exact electric field distribution near the focus of a high numerical aperture (NA) objective lens, accounting for vectorial (polarization) effects that are significant when NA > 0.7.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from monte_carlo import RichardsWolfSimulator

sns.set_theme(style="whitegrid", font_scale=1.5)

import os
output_dir = '../data/richards_wolf'
os.makedirs(output_dir, exist_ok=True)

## Setup: High-NA System

In [ ]:
# System parameters
wavelength = 0.532  # microns (green laser)
NA = 0.9  # High NA objective
n_medium = 1.0  # Air

# Create simulator
rw = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA,
    n_medium=n_medium,
    polarization='x'
)

print(f"Wavelength: {wavelength} μm")
print(f"NA: {NA}")
print(f"Airy radius: {rw.airy_radius:.4f} μm")
print(f"Angular aperture: {np.degrees(rw.angular_aperture):.2f}°")

## Focal Plane Intensity (z=0)

Calculate the intensity distribution in the focal plane.

In [ ]:
# Create radial coordinate array
r_max = 3 * rw.airy_radius
r = np.linspace(0, r_max, 100)

# Compute intensity along x-axis (y=0)
x = r
y = np.zeros_like(r)

print("Computing Richards-Wolf intensity...")
intensity = rw.focal_plane_intensity_pattern(x, y)

# Also compute field components to see vectorial effects
z = np.zeros_like(r)
Ex, Ey, Ez = rw.compute_field(r, z)

print(f"Computed {len(r)} points")
print(f"Peak intensity: {intensity.max():.4f}")
print(f"Ex/Ez ratio at center: {np.abs(Ex[0])/np.abs(Ez[0]):.2f}")

## Plot: Focal Plane Intensity Profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Intensity profile
ax = axes[0]
ax.plot(r, intensity, linewidth=3, color='tab:blue', label='Richards-Wolf')
ax.axvline(rw.airy_radius, color='gray', linestyle='--', alpha=0.5, 
           label=f'Airy radius ({rw.airy_radius:.3f} μm)')
ax.set_xlabel('Radial distance r (μm)')
ax.set_ylabel('Normalized Intensity')
ax.set_title('Focal Plane Intensity (z=0)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Field component intensities
ax = axes[1]
ax.plot(r, np.abs(Ex)**2, linewidth=3, label='|Ex|²', alpha=0.7)
ax.plot(r, np.abs(Ey)**2, linewidth=3, label='|Ey|²', alpha=0.7)
ax.plot(r, np.abs(Ez)**2, linewidth=3, label='|Ez|²', alpha=0.7)
ax.set_xlabel('Radial distance r (μm)')
ax.set_ylabel('Field Intensity')
ax.set_title('Vector Field Components (x-polarized)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/focal_plane_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Note: Ez component is non-zero due to high NA (vectorial effects)")

## 2D Intensity Distribution

Create a 2D intensity map in the focal plane.

In [ ]:
# Create 2D grid
nx, ny = 100, 100
x_range = 2 * rw.airy_radius
x_grid = np.linspace(-x_range, x_range, nx)
y_grid = np.linspace(-x_range, x_range, ny)
X, Y = np.meshgrid(x_grid, y_grid)

print(f"Computing 2D intensity map ({nx}x{ny} points)...")
print("This may take a few minutes...")

# Compute intensity for each point
intensity_2d = rw.focal_plane_intensity_pattern(X.flatten(), Y.flatten())
intensity_2d = intensity_2d.reshape(X.shape)

print("Done!")

In [ ]:
# Plot 2D intensity map
fig, ax = plt.subplots(1, 1, figsize=(10, 9))

im = ax.imshow(intensity_2d, extent=[-x_range, x_range, -x_range, x_range],
               origin='lower', cmap='hot', aspect='auto')
plt.colorbar(im, ax=ax, label='Normalized Intensity')
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.set_title(f'Richards-Wolf Focal Plane Intensity (NA={NA})', fontweight='bold')

# Add Airy disk circle for reference
circle = plt.Circle((0, 0), rw.airy_radius, fill=False, 
                     color='cyan', linewidth=2, label='Airy radius')
ax.add_patch(circle)
ax.legend()

plt.tight_layout()
plt.savefig(f'{output_dir}/focal_plane_2d.png', dpi=150, bbox_inches='tight')
plt.show()

## Axial Intensity (along z-axis)

Examine the intensity distribution along the optical axis (r=0).

In [ ]:
# Axial coordinates
z_range = 3  # microns
z_axis = np.linspace(-z_range, z_range, 100)
r_axis = np.zeros_like(z_axis)

print("Computing axial intensity...")
Ex_axis, Ey_axis, Ez_axis = rw.compute_field(r_axis, z_axis)
intensity_axis = np.abs(Ex_axis)**2 + np.abs(Ey_axis)**2 + np.abs(Ez_axis)**2
intensity_axis = intensity_axis / np.max(intensity_axis)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot(z_axis, intensity_axis, linewidth=3, color='tab:red')
ax.axvline(0, color='gray', linestyle='--', alpha=0.5, label='Focal plane')
ax.set_xlabel('Axial position z (μm)')
ax.set_ylabel('Normalized Intensity')
ax.set_title('Axial Intensity Distribution (r=0)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/axial_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

The Richards-Wolf simulator successfully computes:
- Vectorial electric field components (Ex, Ey, Ez)
- Total intensity distributions
- Proper handling of high-NA vectorial effects (Ez ≠ 0)

This can be used to generate realistic intensity patterns for Monte Carlo photon sampling in high-NA systems.